# pybtex_docutils

pybtex_docutils is a pybtex backend that turns formatted bibliography entries into docutils nodes (paragraphs,
citations, footnotes and the references to them). sphinxcontrib-bibtex renders bibliographies in Sphinx through it, and
for plain docutils it offers a `simplebibliography` directive, which a caller registers itself. These cells cover the
plug-in registration, each node kind, and a whole reStructuredText document rendered to HTML.

In [ ]:
# Example 1: it registers itself as pybtex's `docutils` backend, so pybtex finds it by name.
import importlib.metadata

import pybtex_docutils
from pybtex.plugin import find_plugin

names = [ep.name for ep in importlib.metadata.entry_points(group="pybtex.backends")]
print(names)
assert "docutils" in names
assert find_plugin("pybtex.backends", "docutils") is pybtex_docutils.Backend

In [ ]:
# Example 2: each formatted entry becomes a docutils paragraph node.
import docutils.nodes
from pybtex.database import parse_string

BIB = r"""@book{knuth1984, author = {Knuth, Donald E.}, title = {The {\TeX}book}, publisher = {Addison-Wesley}, year = {1984}}
@article{godel1931, author = {G{\"o}del, Kurt}, title = {{\"U}ber formal unentscheidbare S{\"a}tze},
  journal = {Monatshefte f{\"u}r Mathematik und Physik}, volume = {38}, pages = {173--198}, year = {1931}}
@article{einstein1905, author = {Einstein, Albert and von Neumann, John}, title = {Zur Elektrodynamik bewegter K{\"o}rper},
  journal = {Annalen der Physik}, volume = {17}, year = {1905}}
"""
data = parse_string(BIB, "bibtex")
style = find_plugin("pybtex.style.formatting", "plain")()
backend = pybtex_docutils.Backend()
paragraphs = [backend.paragraph(entry) for entry in style.format_entries(data.entries.values())]
for node in paragraphs:
    print(node.astext())
assert all(isinstance(node, docutils.nodes.paragraph) for node in paragraphs)
assert paragraphs[1].astext().startswith("Kurt Gödel. Über formal unentscheidbare")

In [ ]:
# Example 3: citation and footnote nodes, and the references that point at them, inside a docutils document.
import docutils.utils
from docutils.frontend import get_default_settings
from docutils.parsers.rst import Parser

document = docutils.utils.new_document("refs.rst", get_default_settings(Parser))
entries = list(find_plugin("pybtex.style.formatting", "unsrt")().format_entries(data.entries.values()))
citation, reference = backend.citation(entries[0], document), backend.citation_reference(entries[0], document)
print(citation.astext().splitlines()[0], "|", reference.astext())
assert isinstance(citation, docutils.nodes.citation) and citation["names"] == ["knuth1984"]
assert isinstance(reference, docutils.nodes.citation_reference) and reference["refname"] == "knuth1984"

footnote, footnote_ref = backend.footnote(entries[1], document), backend.footnote_reference(entries[1], document)
print(footnote["names"], footnote["auto"], footnote_ref["refname"])
assert isinstance(footnote, docutils.nodes.footnote) and footnote["names"] == ["godel1931"] and footnote["auto"] == 1
assert isinstance(footnote_ref, docutils.nodes.footnote_reference) and footnote_ref["refname"] == "godel1931"

In [ ]:
# Example 4: a whole document: register the directive, cite two entries, render HTML with linked citations.
import pathlib, tempfile

from docutils.core import publish_string
from docutils.parsers.rst import directives

work = pathlib.Path(tempfile.mkdtemp(prefix="pybtex-docutils-nb-"))
(work / "refs.bib").write_text(BIB)
source = "See [knuth1984]_ and [godel1931]_.\n\n.. simplebibliography:: refs.bib\n   :style: plain\n"
directives.register_directive("simplebibliography", pybtex_docutils.SimpleBibliography)
html = publish_string(source, source_path=str(work / "index.rst"), writer_name="html5",
                      settings_overrides={"report_level": 2, "halt_level": 2}).decode()
assert 'class="citation-reference" href="#knuth1984"' in html and 'href="#godel1931"' in html
assert '<div class="citation" id="godel1931"' in html
assert "Kurt Gödel. Über formal unentscheidbare sätze. <em>Monatshefte für Mathematik und Physik</em>" in html
print([line for line in html.splitlines() if "citation" in line][:4])

In [ ]:
# Example 5: what fails, and how: a missing .bib file is a document error, and importing the module alone
# does not register the directive (a caller does, as in Example 4).
import subprocess, sys

from docutils.utils import SystemMessage

try:
    publish_string(".. simplebibliography:: missing.bib\n", source_path=str(work / "other.rst"), writer_name="html5",
                   settings_overrides={"report_level": 5, "halt_level": 2})
except SystemMessage as error:
    print("missing file:", error)
    assert "could not open bibtex file" in str(error)
else:
    raise AssertionError("a missing .bib file was accepted")

probe = "import pybtex_docutils\nfrom docutils.core import publish_string\npublish_string('.. simplebibliography:: x.bib\\n', settings_overrides={'report_level': 5, 'halt_level': 2})"
fresh = subprocess.run([sys.executable, "-c", probe], capture_output=True, text=True, timeout=120)
print("fresh interpreter exit", fresh.returncode, "- unknown directive:", 'Unknown directive type "simplebibliography"' in fresh.stderr)
assert fresh.returncode != 0 and 'Unknown directive type "simplebibliography"' in fresh.stderr, fresh.stderr[-500:]